# ChurnSignal — Milestone 8: Retention Intelligence & Business Simulation
**Learning-First SaaS Customer Retention Intelligence**

Welcome to **Milestone 8**! In the previous milestones, we built a regularized LightGBM champion model ($P(\text{churn})$) and explained its underlying factors with TreeSHAP.

However, in production SaaS environments, **machine learning predictions alone do not retain customers—targeted business interventions do**.

This milestone bridges the gap between statistical modeling and Customer Success (CS) operations:
1. **The Transparent Customer Health Score (0–100)**: A deterministic operational vitality metric.
2. **Expected Revenue at Risk**: Weighting predictions by financial exposure ($\text{MRR} \times P(\text{churn})$).
3. **The Strategic 2x2 Retention Matrix**: Cross-tabulating operational health against predictive churn risk.
4. **Contextual Retention Playbooks**: Automated translation from local SHAP drivers into immediate CS action workflows.
5. **Counterfactual What-If Simulator**: Exploring how behavioral shifts impact model predictions with explicit causal safeguards.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Add project root to sys.path
project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.loader import load_prediction_eligible_data
from src.features.engineer import engineer_all_features
from src.features.pipeline import chronological_split, TARGET_COLUMN
from src.models.baselines import load_model
from src.explainability.explainer import build_tree_explainer, explain_single_customer
from src.explainability.retention import (
    calculate_customer_health_score,
    assign_risk_tier,
    calculate_expected_mrr_at_risk,
    generate_retention_playbook,
    simulate_what_if_counterfactual,
    plot_retention_matrix
)

## 1. Load Preprocessed Data & Champion Artifacts

We load our pipeline, champion LightGBM model, and validation partition to score active accounts.

In [ ]:
import joblib

# Load and preprocess
eligible_df = load_prediction_eligible_data()
featured_df = engineer_all_features(eligible_df)
train_df, val_df, test_df = chronological_split(featured_df)

# Load artifacts
pipeline = joblib.load(project_root / "artifacts" / "feature_pipeline.joblib")
champion_model = joblib.load(project_root / "artifacts" / "champion_model.joblib")
feature_names = pipeline.get_feature_names()

# Transform validation split
X_val = pipeline.transform(val_df)
y_val = val_df[TARGET_COLUMN].to_numpy()
val_probs = champion_model.predict_proba(X_val)[:, 1]

print(f"Loaded validation cohort: {len(val_df):,} accounts ready for retention intelligence scoring.")

## 2. Customer Health Score (0–100) vs ML Churn Probability

A common misconception is that machine learning makes operational metrics obsolete. In reality, they serve complementary purposes:

| Attribute | Customer Health Score (0–100) | ML Churn Probability (0–100%) |
|---|---|---|
| **Nature** | Deterministic, rule-based operational index | Calibrated statistical risk estimate |
| **Pillars** | Engagement (35%), Sentiment (25%), Friction (20%), Billing (20%) | 30 non-linear engineered features + interactions |
| **Time Horizon** | Current account vitality / state | Forward-looking 30-day exit prediction |
| **Interpretation** | "How healthy is this customer right now?" | "What is the probability this customer will cancel next month?" |


In [ ]:
# Compute health scores
health_scores = calculate_customer_health_score(val_df).to_numpy()

val_scored = val_df.copy()
val_scored["churn_probability"] = val_probs
val_scored["health_score"] = health_scores
val_scored["risk_tier"] = [assign_risk_tier(p) for p in val_probs]
val_scored["expected_mrr_at_risk"] = [
    calculate_expected_mrr_at_risk(mrr, p) for mrr, p in zip(val_df["mrr"], val_probs)
]

print("Operational Risk Tiers across Validation Cohort:")
print(val_scored["risk_tier"].value_counts(normalize=True).mul(100).round(1).astype(str) + "%")

## 3. Financial Prioritization: Expected MRR at Risk

If a Customer Success team only calls accounts with the highest raw churn probability, they waste time calling low-value accounts with negligible revenue impact.

$$\text{Expected MRR at Risk} = \text{MRR} \times P(\text{churn})$$

Let's inspect the top accounts ranked by financial exposure:

In [ ]:
# Rank by Expected MRR at Risk
top_financial_risk = val_scored.sort_values(by="expected_mrr_at_risk", ascending=False).head(10)

cols_display = ["user_id", "plan_type", "mrr", "churn_probability", "health_score", "expected_mrr_at_risk"]
display(top_financial_risk[cols_display])

print(f"Total Cohort MRR: ${val_scored['mrr'].sum():,.2f}")
print(f"Total Expected Monthly Churn Loss: ${val_scored['expected_mrr_at_risk'].sum():,.2f}")

## 4. The Strategic 2x2 Retention & Health Matrix

By plotting Customer Health against Predicted Churn Probability, accounts naturally separate into four actionable quadrants:
1. **Crisis Zone (Low Health, High Risk)**: Severely disengaged, high probability of leaving. Requires immediate multi-channel intervention.
2. **Blindspot Churn (High Health, High Risk)**: Happy, high-usage accounts that are suddenly at risk (e.g. billing failures, expiring cards, unresolved platform incidents).
3. **Passive / Zombie (Low Health, Low Risk)**: Long-tenured accounts with low usage that haven't cancelled yet. Prime targets for proactive adoption campaigns.
4. **Champion Accounts (High Health, Low Risk)**: Flourishing accounts with negligible churn risk. Ideal for expansion, upselling, and advocacy.

In [ ]:
# Display Figure 15: Strategic 2x2 Retention Matrix
display(Image(filename=str(project_root / "reports" / "figures" / "15_retention_matrix.png")))

## 5. Automated Contextual Retention Playbooks

Rather than issuing generic warnings, ChurnSignal synthesizes the account's local SHAP risk drivers into concrete playbooks:

In [ ]:
# Generate playbook for the top financial risk account
top_acct = top_financial_risk.iloc[0]
cust_idx = val_df.index.get_loc(top_acct.name)

explainer = build_tree_explainer(champion_model)
cust_explanation = explain_single_customer(explainer, X_val[cust_idx], feature_names, top_n=3)

playbook = generate_retention_playbook(
    cust_explanation["risk_drivers"],
    churn_prob=top_acct["churn_probability"],
    mrr=top_acct["mrr"],
    health_score=top_acct["health_score"]
)

print(f"Account: {top_acct['user_id']} | Plan: {top_acct['plan_type']} | MRR: ${top_acct['mrr']:.2f}")
print(f"Risk Tier: {playbook['risk_tier']} | Expected MRR at Risk: ${playbook['expected_mrr_at_risk']:.2f}")
print(f"Customer Health Score: {top_acct['health_score']:.1f}/100\n")

print("Recommended CS Action Playbooks:")
for idx, p in enumerate(playbook["recommended_playbooks"], 1):
    print(f"Playbook {idx}: {p['playbook']}")
    print(f"  • Urgency: {p['urgency']}")
    print(f"  • Channel: {p['channel']}")
    print(f"  • Action:  {p['action']}\n")

## 6. Counterfactual What-If Simulator

The What-If Simulator allows CS teams and account executives to test hypotheses:
*"If we help this customer resolve open tickets and increase their feature usage, what does the model estimate their churn probability will become?"*

> **IMPORTANT CAUSAL SAFEGUARD**:
> Model simulations measure **associational sensitivity** $P(Y \mid X_{\text{mod}})$, not proven **causal interventions** $P(Y \mid \text{do}(X))$. 
> Artificially boosting a metric (e.g. sending automated spam clicks) does not solve root customer value issues. The simulator is a decision-support guide, not a guarantee.

In [ ]:
# Simulate What-If on top account
target_row = val_df.loc[top_acct.name]

# Define hypothetical intervention: resolve tickets, clear billing issues, boost usage
modifications = {
    "support_tickets": 0,
    "payment_failures": 0,
    "feature_usage_score": min(100.0, float(target_row["feature_usage_score"]) + 25.0)
}

sim_output = simulate_what_if_counterfactual(
    champion_model,
    pipeline,
    target_row,
    modifications
)

print(f"Account: {top_acct['user_id']}")
print(f"Base Churn Probability:      {sim_output['base_churn_probability']:.2%}")
print(f"Simulated Churn Probability: {sim_output['simulated_churn_probability']:.2%}")
print(f"Estimated Probability Delta: {sim_output['probability_delta']:+.2%}")
print(f"Base Health Score:           {sim_output['base_health_score']:.1f}")
print(f"Simulated Health Score:      {sim_output['simulated_health_score']:.1f} ({sim_output['health_score_delta']:+.1f})\n")
print(f"Causal Safeguard: {sim_output['causal_disclaimer']}")